In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dinmkeljiame/doctamper/DocTamperV1-TrainingSet/lock.mdb
/kaggle/input/datasets/dinmkeljiame/doctamper/DocTamperV1-TrainingSet/data.mdb
/kaggle/input/datasets/dinmkeljiame/doctamper/DocTamperV1-TestingSet/lock.mdb
/kaggle/input/datasets/dinmkeljiame/doctamper/DocTamperV1-TestingSet/data.mdb
/kaggle/input/datasets/dinmkeljiame/doctamper/DocTamperV1-SCD/lock.mdb
/kaggle/input/datasets/dinmkeljiame/doctamper/DocTamperV1-SCD/data.mdb
/kaggle/input/datasets/dinmkeljiame/doctamper/DocTamperV1-FCD/lock.mdb
/kaggle/input/datasets/dinmkeljiame/doctamper/DocTamperV1-FCD/data.mdb


In [2]:
!pip install -q lmdb jpegio opencv-python-headless seaborn six segmentation-models-pytorch==0.3.4 albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 MB 25.5 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.3 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda

In [3]:
"""
MDAV — DocTamper tamper-localization training (Kaggle free-tier friendly)
=========================================================================
Corrected build. Changes vs the previous version:
  * Host-RAM safety: pin_memory=False, train loader persistent_workers=False
    (so train + val workers never coexist at the epoch boundary), uint8 DCT
    transport (8x smaller than int64 across every prefetched batch).
  * Resume safety: OneCycleLR replaced with a stateless warmup+cosine schedule
    computed purely from (epoch, intra-epoch step). Mid-session crashes are the
    norm on Kaggle; this makes re-running a partial epoch reproduce the exact
    same LR curve instead of drifting the schedule out of sync. Nothing
    schedule-related lives in the checkpoint anymore, so it cannot desync.
  * Validation cleanup: single empty_cache() after the loop, not per-iter
    (the per-iter version forced a device sync and slowed validation; it also
    never addressed the host-RAM crash, which is a CPU-side issue).
  * Optional quality fix: 8-aligned random crop in training (was fixed
    top-left, which only ever showed the model one corner of larger docs).
"""

import os, io, time, math, tempfile, threading
from dataclasses import dataclass

import cv2
import lmdb
import six
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import segmentation_models_pytorch as smp


# ============================== CONFIG =======================================
@dataclass
class Cfg:
    # paths — Kaggle mounted dataset structure
    train_lmdb: str = "/kaggle/input/datasets/dinmkeljiame/doctamper/DocTamperV1-TrainingSet"
    val_lmdb:   str = "/kaggle/input/datasets/dinmkeljiame/doctamper/DocTamperV1-TestingSet"
    ckpt_in:    str = "/kaggle/input/mdav-ckpt/ckpt.pth"   # stale epochs=6 ckpt? start fresh instead
    ckpt_out:   str = "/kaggle/working/ckpt.pth"
    best_out:   str = "/kaggle/working/best.pth"

    # model
    use_dct: bool = True
    dct_dim: int = 8
    encoder: str = "resnet18"       # smp adapts the stem to the 11-ch (RGB + 8 DCT) input
    classes: int = 2

    # data
    img_size: int = 512
    minq: int = 75
    max_train: int = 10000
    max_val: int = 1000
    num_workers: int = 2            # host-RAM lever; raise cautiously if RAM stays low

    # optim
    epochs: int = 20               # DO NOT change once a ckpt exists — the schedule
                                    # is derived from this; mid-cycle changes shift the LR curve
    batch: int = 24                 # if the host-RAM meter still spikes near 30 GiB, drop to 16
    accum: int = 1
    lr: float = 2.0e-4
    lr_min: float = 1e-6
    warmup_frac: float = 0.1
    wd: float = 1e-2
    amp: bool = True

    # session control
    max_seconds: int = 11 * 3600    # Kaggle GPU session ~12h; margin left for the final ckpt.
                                    # Weekly quota is ~30h though, so ~3 sessions max per week.
    save_every_steps: int = 500
    seed: int = 42


# ============================== DATASET ======================================
_QUALITIES = (75, 80, 85, 90)

class DocTamperLMDB(Dataset):
    def __init__(self, root, img_size=512, minq=75, train=True, max_nums=None):
        self.root, self.img_size, self.minq, self.train = root, img_size, minq, train

        # Open briefly just to read the sample count, then close. The real env is
        # opened lazily per-worker in _init_env to avoid multiprocessing lock issues.
        env = lmdb.open(root, readonly=True, lock=False)
        with env.begin(write=False) as txn:
            self.n = int(txn.get(b'num-samples'))
        env.close()

        self.env = None
        self.n = self.n if max_nums is None else min(max_nums, self.n)
        self._tmp = tempfile.mkdtemp(prefix="dct_")
        self.norm = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])

    def _init_env(self):
        if self.env is None:
            self.env = lmdb.open(self.root, max_readers=64, readonly=True,
                                 lock=False, readahead=False, meminit=False)

    def __len__(self):
        return self.n

    def _qualities_for(self, index):
        if not self.train:
            return [self.minq]
        rng = np.random.default_rng()
        rounds = int(rng.integers(1, 4))
        qs = [int(rng.choice([q for q in _QUALITIES if q >= self.minq])) for _ in range(rounds)]
        return qs

    def _recompress_dct(self, pil_L, qualities):
        im = pil_L
        for q in qualities[:-1]:                          # intermediate recompressions in-memory
            buf = io.BytesIO(); im.save(buf, "JPEG", quality=int(q)); buf.seek(0)
            im = Image.open(buf); im.load()

        # jpegio needs a real file path, so the final pass goes through /tmp.
        path = os.path.join(self._tmp, f"{os.getpid()}_{threading.get_ident()}.jpg")
        im.save(path, "JPEG", quality=int(qualities[-1]))
        im = Image.open(path); im.load()

        import jpegio
        jpg = jpegio.read(path)
        # values land in [0, 20]; uint8 keeps the transported tensor 8x smaller than int64.
        dct = np.clip(np.abs(jpg.coef_arrays[0]), 0, 20).astype(np.uint8)

        if os.path.exists(path):
            os.remove(path)

        return im.convert("RGB"), dct

    def __getitem__(self, index):
        self._init_env()
        with self.env.begin(write=False) as txn:
            imgbuf = txn.get(('image-%09d' % index).encode('utf-8'))
            lblbuf = txn.get(('label-%09d' % index).encode('utf-8'))

        buf = six.BytesIO(); buf.write(imgbuf); buf.seek(0)
        im = Image.open(buf).convert("L")
        mask = (cv2.imdecode(np.frombuffer(lblbuf, np.uint8), 0) != 0).astype(np.int64)

        if self.train and np.random.rand() < 0.5:        # horizontal flip
            im = im.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask[:, ::-1].copy()

        rgb, dct = self._recompress_dct(im, self._qualities_for(index))

        # ---- 8-aligned crop -------------------------------------------------
        # Offsets MUST be multiples of 8 or the DCT block grid stops lining up
        # with the RGB/mask pixels and the DCT branch sees garbage.
        H = min(dct.shape[0], mask.shape[0], rgb.size[1])
        W = min(dct.shape[1], mask.shape[1], rgb.size[0])
        h = min(self.img_size, H)
        w = min(self.img_size, W)
        if self.train and (H > h or W > w):
            top  = 8 * int(np.random.randint(0, (H - h) // 8 + 1))
            left = 8 * int(np.random.randint(0, (W - w) // 8 + 1))
        else:
            top = left = 0
        rgb  = rgb.crop((left, top, left + w, top + h))
        dct  = dct[top:top + h, left:left + w]
        mask = mask[top:top + h, left:left + w]

        return {
            "image": self.norm(rgb),
            "dct":   torch.from_numpy(np.ascontiguousarray(dct)),   # uint8
            "label": torch.from_numpy(mask),                        # int64 (CE target)
        }


# ============================== MODEL ========================================
class DCTEmbed(nn.Module):
    def __init__(self, n_bins=21, dim=8):
        super().__init__()
        self.emb = nn.Embedding(n_bins, dim)

    def forward(self, dct):                               # dct: (B, H, W) uint8
        return self.emb(dct.long()).permute(0, 3, 1, 2).contiguous()   # -> (B, dim, H, W)


class MDAVNet(nn.Module):
    def __init__(self, cfg: Cfg):
        super().__init__()
        self.dct = DCTEmbed(21, cfg.dct_dim) if cfg.use_dct else None
        in_ch = 3 + (cfg.dct_dim if cfg.use_dct else 0)
        self.net = smp.Unet(
            encoder_name=cfg.encoder,
            encoder_weights="imagenet",
            in_channels=in_ch,
            classes=cfg.classes,
        )

    def forward(self, image, dct=None):
        x = image if self.dct is None else torch.cat([image, self.dct(dct)], dim=1)
        return self.net(x)


# ============================== LOSS / METRIC ================================
class ComboLoss(nn.Module):
    def __init__(self, fg_weight=3.0):
        super().__init__()
        # weight is a registered buffer, so .to(dev) on the module moves it correctly.
        self.ce = nn.CrossEntropyLoss(weight=torch.tensor([1.0, fg_weight]))

    def forward(self, logits, target):
        ce = self.ce(logits, target)
        prob = logits.softmax(1)[:, 1]
        t = (target == 1).float()
        inter = (prob * t).sum((1, 2))
        dice = 1 - (2 * inter + 1) / (prob.sum((1, 2)) + t.sum((1, 2)) + 1)
        return ce + dice.mean()


@torch.no_grad()
def tampered_f1(logits, target):
    pred = logits.argmax(1)
    tp = ((pred == 1) & (target == 1)).sum().item()
    fp = ((pred == 1) & (target == 0)).sum().item()
    fn = ((pred == 0) & (target == 1)).sum().item()
    return tp, fp, fn


# ============================== LR SCHEDULE ==================================
def lr_at(epoch, local_step, steps_per_epoch, cfg: Cfg):
    """Stateless warmup + cosine, a pure function of training position.

    Because it depends only on (epoch, local_step) and not on a running counter,
    re-running a partial epoch after a crash retraces the identical curve — no
    drift, nothing to restore from the checkpoint.
    """
    total = cfg.epochs * steps_per_epoch
    step = min(epoch * steps_per_epoch + local_step, total - 1)
    warm = max(1, int(cfg.warmup_frac * total))
    if step < warm:
        return cfg.lr * (step + 1) / warm
    prog = (step - warm) / max(1, total - warm)
    return cfg.lr_min + 0.5 * (cfg.lr - cfg.lr_min) * (1 + math.cos(math.pi * prog))


# ============================== CHECKPOINT ===================================
def save_ckpt(path, model, opt, scaler, epoch, step, best):
    tmp = path + ".tmp"
    torch.save({
        "model": model.state_dict(), "opt": opt.state_dict(),
        "scaler": scaler.state_dict() if scaler else None,
        "epoch": epoch, "step": step, "best": best,
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }, tmp)
    os.replace(tmp, path)   # atomic: a crash mid-write can't corrupt the live ckpt


def load_ckpt(path, model, opt, scaler):
    if not path or not os.path.exists(path):
        return 0, 0, -1.0
    s = torch.load(path, map_location="cuda" if torch.cuda.is_available() else "cpu",
                   weights_only=False)
    model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"])
    if scaler and s.get("scaler"):
        scaler.load_state_dict(s["scaler"])
    torch.set_rng_state(s["torch_rng"].cpu())
    if s.get("cuda_rng") is not None and torch.cuda.is_available():
        try:
            torch.cuda.set_rng_state_all(s["cuda_rng"])
        except Exception:
            pass
    print(f"resumed @ epoch {s['epoch']} step {s['step']} best {s['best']:.4f}")
    # start_epoch = last completed epoch + 1; partial epochs are re-run from scratch.
    return s["epoch"] + 1, s["step"], s["best"]


# ============================== TRAIN ========================================
def run(cfg: Cfg):
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    tr = DocTamperLMDB(cfg.train_lmdb, cfg.img_size, cfg.minq, train=True,  max_nums=cfg.max_train)
    va = DocTamperLMDB(cfg.val_lmdb,   cfg.img_size, cfg.minq, train=False, max_nums=cfg.max_val)

    # pin_memory=False  -> no non-reclaimable page-locked buffers (host-RAM safety)
    # persistent_workers=False on train -> train workers exit before val workers spawn,
    #                                      so the two sets never coexist at the boundary.
    trl = DataLoader(tr, batch_size=cfg.batch, shuffle=True,  num_workers=cfg.num_workers,
                     pin_memory=False, prefetch_factor=2, drop_last=True,
                     persistent_workers=False)
    val = DataLoader(va, batch_size=cfg.batch, shuffle=False, num_workers=cfg.num_workers,
                     pin_memory=False)

    model = MDAVNet(cfg).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    steps_per_epoch = math.ceil(len(trl) / cfg.accum)
    scaler = torch.amp.GradScaler(enabled=cfg.amp)
    crit = ComboLoss().to(dev)

    start_epoch, gstep, best = load_ckpt(cfg.ckpt_in, model, opt, scaler)
    t0 = time.time()

    def out_of_time():
        return time.time() - t0 > cfg.max_seconds

    for epoch in range(start_epoch, cfg.epochs):
        model.train(); opt.zero_grad(set_to_none=True)
        local_step = 0                      # optimizer steps taken *this epoch* (drives LR)
        for it, b in enumerate(trl):
            img = b["image"].to(dev, non_blocking=True)
            dct = b["dct"].to(dev, non_blocking=True)
            lbl = b["label"].to(dev, non_blocking=True)

            with torch.amp.autocast(device_type=dev, enabled=cfg.amp):
                logits = model(img, dct if cfg.use_dct else None)
                loss = crit(logits, lbl) / cfg.accum

            scaler.scale(loss).backward()

            if (it + 1) % cfg.accum == 0:
                for g in opt.param_groups:
                    g["lr"] = lr_at(epoch, local_step, steps_per_epoch, cfg)
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True)
                local_step += 1
                gstep += 1

                if gstep % 50 == 0:
                    print(f"e{epoch} s{gstep} loss {loss.item()*cfg.accum:.4f} "
                          f"lr {opt.param_groups[0]['lr']:.2e}")
                if gstep % cfg.save_every_steps == 0:
                    # emergency mid-epoch save: epoch-1 so resume re-runs this epoch cleanly
                    save_ckpt(cfg.ckpt_out, model, opt, scaler, epoch - 1, gstep, best)

            if out_of_time():
                save_ckpt(cfg.ckpt_out, model, opt, scaler, epoch - 1, gstep, best)
                print("wall-clock guard hit — checkpointed and exiting.")
                return

        # ---- validation -----------------------------------------------------
        model.eval(); TP = FP = FN = 0
        with torch.no_grad():
            for b in val:
                img = b["image"].to(dev); dct = b["dct"].to(dev); lbl = b["label"].to(dev)
                with torch.amp.autocast(device_type=dev, enabled=cfg.amp):
                    logits = model(img, dct if cfg.use_dct else None)
                tp, fp, fn = tampered_f1(logits, lbl)
                TP += tp; FP += fp; FN += fn
        if dev == "cuda":
            torch.cuda.empty_cache()        # once per epoch, not per batch

        prec = TP / (TP + FP + 1e-9); rec = TP / (TP + FN + 1e-9)
        f1 = 2 * prec * rec / (prec + rec + 1e-9); iou = TP / (TP + FP + FN + 1e-9)
        print(f"[val e{epoch}] F1 {f1:.4f}  P {prec:.4f}  R {rec:.4f}  IoU {iou:.4f}")

        save_ckpt(cfg.ckpt_out, model, opt, scaler, epoch, gstep, best)
        if f1 > best:
            best = f1
            save_ckpt(cfg.best_out, model, opt, scaler, epoch, gstep, best)
            print(f"  new best F1 {best:.4f} -> {cfg.best_out}")

    print(f"done. best F1 {best:.4f}")


if __name__ == "__main__":
    run(Cfg())

Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 188MB/s]


e0 s50 loss 1.4864 lr 1.20e-05
e0 s100 loss 1.3893 lr 2.40e-05
e0 s150 loss 1.2636 lr 3.61e-05
e0 s200 loss 1.1890 lr 4.81e-05
e0 s250 loss 1.1834 lr 6.01e-05
e0 s300 loss 1.1064 lr 7.21e-05
e0 s350 loss 1.0968 lr 8.41e-05
e0 s400 loss 1.0707 lr 9.62e-05
[val e0] F1 0.2177  P 0.1916  R 0.2520  IoU 0.1221
  new best F1 0.2177 -> /kaggle/working/best.pth
e1 s450 loss 1.0754 lr 1.08e-04
e1 s500 loss 1.0352 lr 1.20e-04
e1 s550 loss 1.0666 lr 1.32e-04
e1 s600 loss 1.0362 lr 1.44e-04
e1 s650 loss 1.0826 lr 1.56e-04
e1 s700 loss 1.0589 lr 1.68e-04
e1 s750 loss 1.0602 lr 1.80e-04
e1 s800 loss 1.0411 lr 1.92e-04
[val e1] F1 0.2128  P 0.1388  R 0.4557  IoU 0.1190
e2 s850 loss 1.0257 lr 2.00e-04
e2 s900 loss 1.0056 lr 2.00e-04
e2 s950 loss 0.9855 lr 2.00e-04
e2 s1000 loss 1.0591 lr 2.00e-04
e2 s1050 loss 0.9798 lr 2.00e-04
e2 s1100 loss 1.0526 lr 1.99e-04
e2 s1150 loss 0.9770 lr 1.99e-04
e2 s1200 loss 1.1401 lr 1.99e-04
[val e2] F1 0.2907  P 0.2497  R 0.3479  IoU 0.1701
  new best F1 0.2907 -> /k